## Study Case Siemenens:

Suponhamos que você tenha sido contratado pela Siemens Healthineers e é responsável por acompanhar a evolução da conectividade dos equipamentos médicos a nível Brasil. O objetivo principal são os equipamentos estarem conectados a uma estrutura de Acesso Remoto e com todos os serviços ativos, e temos dois níveis de conectividade, sendo eles:

- **CONNECTED BUT NOT SRS READY** - quando o equipamento está conectado, mas não está com todos os serviços ativos.
- **SRS READY** – quando o equipamento está conectado e com todos os serviços ativos.

Como uma tarefa mensal, você é responsável por extrair os dados do sistema de controle e analisar a nível Brasil o avanço da conectividade em alinhamento aos Targets, para então direcionar ações de melhorias em prol do atingimento das metas.


### Approach

Para solucionar este problema, estaremos criando um dashboard interativo no streamlit, para que possamos visualizar mensalmente os dados mais importantes e mostrarmos os maiores insights.

In [72]:
import polars as pl
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.functions as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [73]:
spark = SparkSession.builder \
    .appName("Study Case Siemens") \
    .getOrCreate()

In [74]:
workbook_path = "../data/Study Case - Vaga de Estágio CS SO OPE_BRUNO SANTIAGO DA CO.xlsx"
overview_csv = "../data/overview.csv"
target_csv = "../data/target.csv"

pl.read_excel(workbook_path, sheet_name="Overview", engine="calamine").write_csv(overview_csv)
pl.read_excel(workbook_path, sheet_name="Target", engine="calamine").write_csv(target_csv)

df_overview = spark.read.option("header", True).option("inferSchema", True).csv(overview_csv)
df_target = spark.read.option("header", True).option("inferSchema", True).csv(target_csv)

In [75]:
df_overview.describe().show()

+-------+-----------------------+--------------------+---------+-----------+-------------+----------+-----+------------------+---------------------------+
|summary|EQUIPMENT SERIAL NUMBER| LEVEL OF CONNECTION|SYSTEM ID|SYSTEM NAME|BUSINESS LINE|      CITY|STATE|    SERVICE REGION|CURRENT CONNECTIVITY STATUS|
+-------+-----------------------+--------------------+---------+-----------+-------------+----------+-----+------------------+---------------------------+
|  count|                   2286|                2286|     2286|       2286|         2286|      2286| 2286|              2286|                       2286|
|   mean|   7.4773530879868765E9|                NULL|     NULL|       NULL|         NULL|      NULL| NULL|              NULL|                       NULL|
| stddev|   1.501956138698055...|                NULL|     NULL|       NULL|         NULL|      NULL| NULL|              NULL|                       NULL|
|    min|             2100024947|CONNECTED BUT NOT...|   ACU101|   ACU

In [76]:
df_overview.show(5)

+-----------------------+--------------------+---------+-----------+-------------+--------+-----+------------------+---------------------------+---------------------------+
|EQUIPMENT SERIAL NUMBER| LEVEL OF CONNECTION|SYSTEM ID|SYSTEM NAME|BUSINESS LINE|    CITY|STATE|    SERVICE REGION|CURRENT CONNECTIVITY STATUS|EQUIPMENT INSTALLATION DATE|
+-----------------------+--------------------+---------+-----------+-------------+--------+-----+------------------+---------------------------+---------------------------+
|             3049584789|CONNECTED BUT NOT...|   ART111|    ARTIS 1|          AXA| LIMEIRA|   SP|SÃO PAULO INTERIOR|              Connection OK|                 2021-05-13|
|             3024458017|           SRS READY|   SOM157|  SOMATOM 1|           CT|  CACOAL|   RO|            CENTRO|              Connection OK|                 2013-06-11|
|             3030176456|           SRS READY|   SOM165|  SOMATOM 2|           CT| GOIANIA|   GO|            CENTRO|       Connection E

In [77]:
df_target.show(5)

+-----------+-----------+------------------------------+------------------+
|Material ID|SYSTEM NAME|MOBILE SYSTEM OR FIXED SYSTEM?|TARGET (SRS READY)|
+-----------+-----------+------------------------------+------------------+
|     ACU101|   ACUSON 1|                 MOBILE SYSTEM|              0.25|
|     ACU102|   ACUSON 2|                 MOBILE SYSTEM|              0.25|
|     ACU103|   ACUSON 3|                 MOBILE SYSTEM|              0.25|
|     ACU104|   ACUSON 4|                 MOBILE SYSTEM|              0.25|
|     ACU105|   ACUSON 5|                 MOBILE SYSTEM|              0.25|
+-----------+-----------+------------------------------+------------------+
only showing top 5 rows


In [78]:

def rename_cols(df: DataFrame) -> DataFrame:
    for col in df.columns:
        if col != "TARGET (SRS READY)":
            df = df.withColumnRenamed(col, col.replace(" ", "_").lower())
        else:
            df = df.withColumnRenamed(col, "target_srs_ready")

    return df

df_overview = rename_cols(df_overview)
df_target = rename_cols(df_target)

In [79]:
print(df_overview.describe().columns)
print(df_target.describe().columns)

['summary', 'equipment_serial_number', 'level_of_connection', 'system_id', 'system_name', 'business_line', 'city', 'state', 'service_region', 'current_connectivity_status']
['summary', 'material_id', 'system_name', 'mobile_system_or_fixed_system?', 'target_srs_ready']


In [80]:
df_merged = (
    df_overview.alias("t1")
    .join(
        df_target.alias("t2"),
        on=(F.col("t1.system_id") == F.col("t2.material_id")),
        how="left"
    )
)

df_merged = df_merged.select(F.col("t1.*"), F.col("t2.mobile_system_or_fixed_system?"), F.col("t2.target_srs_ready"))
df_merged.show(5)

+-----------------------+--------------------+---------+-----------+-------------+--------+-----+------------------+---------------------------+---------------------------+------------------------------+----------------+
|equipment_serial_number| level_of_connection|system_id|system_name|business_line|    city|state|    service_region|current_connectivity_status|equipment_installation_date|mobile_system_or_fixed_system?|target_srs_ready|
+-----------------------+--------------------+---------+-----------+-------------+--------+-----+------------------+---------------------------+---------------------------+------------------------------+----------------+
|             3049584789|CONNECTED BUT NOT...|   ART111|    ARTIS 1|          AXA| LIMEIRA|   SP|SÃO PAULO INTERIOR|              Connection OK|                 2021-05-13|                  FIXED SYSTEM|            0.87|
|             3024458017|           SRS READY|   SOM157|  SOMATOM 1|           CT|  CACOAL|   RO|            CENTRO|

In [83]:
srs_ready_equipment = df_merged.where(F.col("level_of_connection") == "SRS READY")
srs_ready_adherence = round(srs_ready_equipment.count() / df_merged.count(), 2)
srs_ready_adherence

0.72

In [97]:
srs_by_system = (
    df_merged
    .groupBy(
        "system_id",
        "system_name",
        "target_srs_ready"
    )
    .agg(
        F.count("*").alias("total_equipment"),
        F.sum(
            F.when(F.col("level_of_connection") == "SRS READY", 1)
             .otherwise(0)
        ).alias("srs_ready_equipment")
    )
    .withColumn(
        "srs_ready_pct",
        F.col("srs_ready_equipment") / F.col("total_equipment")
    )
    .withColumn(
        "target_srs_ready_pct",
        F.col("target_srs_ready") * 100
    )
    .withColumn(
        "srs_ready_pct_display",
        F.col("srs_ready_pct") * 100
    )
    .select(
        F.col("system_id"),
        F.col("system_name"),
        F.col("total_equipment"),
        F.col("srs_ready_equipment"),
        F.col("srs_ready_pct_display"),
        F.col("target_srs_ready_pct"),
    ).withColumn(
        "on_target",
        F.when(
            F.col("srs_ready_pct_display") >= F.col("target_srs_ready_pct"),
            1
        ).otherwise(0)
    )
)


srs_by_system.orderBy(F.col("srs_ready_pct_display").desc()).show(truncate=False)

+---------+-----------+---------------+-------------------+---------------------+--------------------+---------+
|system_id|system_name|total_equipment|srs_ready_equipment|srs_ready_pct_display|target_srs_ready_pct|on_target|
+---------+-----------+---------------+-------------------+---------------------+--------------------+---------+
|NUC154   |NUCLEAR 7  |2              |2                  |100.0                |87.0                |1        |
|SOM170   |SOMATOM 7  |2              |2                  |100.0                |87.0                |1        |
|MAG135   |MAGNETON 18|1              |1                  |100.0                |87.0                |1        |
|NUC147   |NUCLEAR 13 |3              |3                  |100.0                |87.0                |1        |
|MAG130   |MAGNETON 13|7              |7                  |100.0                |87.0                |1        |
|SOM162   |SOMATOM 14 |7              |7                  |100.0                |87.0           

In [102]:
not_connected = (
    df_merged.groupby(
        F.col("system_id"),
        F.col("system_name")
    ).agg(
        F.count("*").alias("total_equipment"),
        F.sum(
            F.when(F.col("level_of_connection") == "NOT CONNECTED",1).otherwise(0)
        ).alias("srs_disconnected_equipment")
    )
    .withColumn(
        "disconnected_percentage",
        F.round(F.col("srs_disconnected_equipment") / F.col("total_equipment"), 2)
    )
)

not_connected.orderBy(F.col("disconnected_percentage").desc()).show()

+---------+-----------+---------------+--------------------------+-----------------------+
|system_id|system_name|total_equipment|srs_disconnected_equipment|disconnected_percentage|
+---------+-----------+---------------+--------------------------+-----------------------+
|   ACU102|   ACUSON 2|              2|                         2|                    1.0|
|   ACU109|   ACUSON 9|              2|                         2|                    1.0|
|   ACU101|   ACUSON 1|            157|                       122|                   0.78|
|   ACU104|   ACUSON 4|             71|                        53|                   0.75|
|   MOB141|   MOBILLET|             26|                        19|                   0.73|
|   ARC110|    ARCADIS|             44|                        32|                   0.73|
|   ACU103|   ACUSON 3|              3|                         2|                   0.67|
|   CIO116|       CIOS|             29|                        18|                   0.62|

In [103]:
not_connected = (
    df_merged.groupby(
        F.col("business_line")
    ).agg(
        F.count("*").alias("total_equipment"),
        F.sum(
            F.when(F.col("level_of_connection") == "NOT CONNECTED",1).otherwise(0)
        ).alias("srs_disconnected_equipment")
    )
    .withColumn(
        "disconnected_percentage",
        F.round(F.col("srs_disconnected_equipment") / F.col("total_equipment"), 2)
    )
)

not_connected.orderBy(F.col("disconnected_percentage").desc()).show()

+-------------+---------------+--------------------------+-----------------------+
|business_line|total_equipment|srs_disconnected_equipment|disconnected_percentage|
+-------------+---------------+--------------------------+-----------------------+
|        XP SU|             26|                        19|                   0.73|
|       AXA SU|             73|                        50|                   0.68|
|           US|            320|                       215|                   0.67|
|           CT|            541|                        53|                    0.1|
|        XP WH|            114|                        10|                   0.09|
|           SY|            150|                        11|                   0.07|
|        XP RF|             67|                         4|                   0.06|
|     MI SPECT|            101|                         6|                   0.06|
|           MR|            663|                        33|                   0.05|
|   

In [108]:
print(df_merged.where(
    #(F.col("service_region") == "SÃO PAULO INTERIOR") &
    (F.col("business_line") == "AXA")
    #(F.col("level_of_connection") == "SRS READY")
).count())

183


In [110]:
not_connected = (
    df_merged.groupby(
        F.col("business_line")
    ).agg(
        F.count("*").alias("total_equipment"),
        F.sum(
            F.when(F.col("level_of_connection") == "SRS READY",1).otherwise(0)
        ).alias("srs_ready_equipment")
    )
    .withColumn(
        "ready_pct",
        F.round(F.col("srs_ready_equipment") / F.col("total_equipment"), 2)
    )
)

not_connected.orderBy(F.col("ready_pct").desc()).show()

+-------------+---------------+-------------------+---------+
|business_line|total_equipment|srs_ready_equipment|ready_pct|
+-------------+---------------+-------------------+---------+
|          AXA|            183|                161|     0.88|
|     MI SPECT|            101|                 89|     0.88|
|           MR|            663|                576|     0.87|
|           CT|            541|                458|     0.85|
|       MI PET|             48|                 41|     0.85|
|           SY|            150|                115|     0.77|
|        XP RF|             67|                 50|     0.75|
|        XP WH|            114|                 85|     0.75|
|           US|            320|                 58|     0.18|
|       AXA SU|             73|                 12|     0.16|
|        XP SU|             26|                  1|     0.04|
+-------------+---------------+-------------------+---------+



In [111]:
print(df_merged.select("level_of_connection").where(F.col("level_of_connection") == "SRS READY").count() / df_merged.select("level_of_connection").count())

0.7200349956255468


In [114]:
srs_ready = (
    df_merged.where(
        F.col("level_of_connection") == "SRS READY"
    ).count() / df_merged.count()
)
srs_ready

0.7200349956255468

In [113]:
srs_ready.show()

+--------------------+---------------+-------------------+---------+
| level_of_connection|total_equipment|srs_ready_equipment|ready_pct|
+--------------------+---------------+-------------------+---------+
|CONNECTED BUT NOT...|            229|                  0|      0.0|
|       NOT CONNECTED|            411|                  0|      0.0|
|           SRS READY|           1646|               1646|      1.0|
+--------------------+---------------+-------------------+---------+



In [ ]:
def get_srs_ready_pct(filters: dict[str, str] | None) -> float:
    """Retorna a taxa de equipamentos com conexão SRS ativa."""
    srs_ready = F.col("level_of_connection") == "SRS READY"
    df = df_merged
    if not filters:
        return round((df.filter(srs_ready).count() / df.count()) * 100, 2)

    for col, value in filters.items():
        df = df.filter(F.col(col) == value)

    return round((df.filter(srs_ready).count() / df_merged.count()) * 100, 2)

filters = {
    "service_region": "SÃO PAULO INTERIOR",
    "business_line": "AXA",
}

print(get_srs_ready_pct(filters))

filters = {}

print(get_srs_ready_pct(filters))

filters = {
    "service_region": "RIO DE JANEIRO",
    "business_line": "AXA",
    "level_of_connection": "NOT CONNECTED"
}

print(get_srs_ready_pct(filters))

1.66
72.0
0.0


In [130]:
def get_connected_not_srs(filters: dict[str, str]) -> int:
    """Retorna a quantidade de equipamentos conectados na rede mas que ainda não tem SRS"""
    connected_not_srs = (
        F.col("level_of_connection") == "CONNECTED BUT NOT SRS READY"
    ) & (F.col("current_connectivity_status") == "Connection OK")
    df = df_merged
    if not filters:
        return df_merged.filter(connected_not_srs).count()
    for col, value in filters.items():
        df = df.filter(F.col(col) == value.upper())
    return df.filter(connected_not_srs).count()

filters = {
    "service_region": "SÃO PAULO INTERIOR",
    "business_line": "AXA",
}

print(get_connected_not_srs(filters))

filters = {}

print(get_connected_not_srs(filters))

filters = {
    "service_region": "RIO DE JANEIRO",
    "business_line": "AXA",
    "level_of_connection": "NOT CONNECTED"
}

print(get_connected_not_srs(filters))

1
68
0


In [132]:
filtered_df = {
        "level_of_connection": "SRS READY",
        "business_line": "AXA",
    }

df_merged.filter((F.col("level_of_connection") == "SRS READY") & (F.col("business_line") == "AXA")).count()

161

In [143]:
service_distribution = (
    df_merged.groupby(
        F.col("level_of_connection"),
        F.col("service_region")
    ).agg(F.count("*").alias("equipment_count"))
    .orderBy("level_of_connection")
    .withColumn(
        "service_region",
        F.when(
            F.col("service_region") == "SÃO PAULO INTERIOR",
            F.lit("SP INTERIOR")
        ).otherwise(F.col("service_region"))
    )
)

service_distribution.show()

+--------------------+--------------+---------------+
| level_of_connection|service_region|equipment_count|
+--------------------+--------------+---------------+
|CONNECTED BUT NOT...|   SP INTERIOR|             32|
|CONNECTED BUT NOT...|     SÃO PAULO|             88|
|CONNECTED BUT NOT...|RIO DE JANEIRO|             29|
|CONNECTED BUT NOT...|           SUL|             25|
|CONNECTED BUT NOT...|        CENTRO|             32|
|CONNECTED BUT NOT...|      NORDESTE|             23|
|       NOT CONNECTED|        CENTRO|            134|
|       NOT CONNECTED|RIO DE JANEIRO|             40|
|       NOT CONNECTED|      NORDESTE|             34|
|       NOT CONNECTED|           SUL|             50|
|       NOT CONNECTED|     SÃO PAULO|            106|
|       NOT CONNECTED|   SP INTERIOR|             47|
|           SRS READY|        CENTRO|            301|
|           SRS READY|           SUL|            227|
|           SRS READY|      NORDESTE|            258|
|           SRS READY|RIO DE

In [157]:
import plotly.express as px

color_map = {
    "SRS READY": "#009999",
    "CONNECTED BUT NOT SRS READY": "#ec6602",
    "NOT CONNECTED": "#D05353"
}

fig = px.bar(
    service_distribution,
    x="service_region",
    y="equipment_count",
    color="level_of_connection",
    title="Distribuição por Regional de Serviço",
    color_discrete_map=color_map
)

fig.update_yaxes(visible=False)

fig.update_traces(
    texttemplate='%{y}',
    textposition='outside'    # Options: 'outside', 'inside', 'auto', 'none'
)

fig.update_layout(
    legend={
        "orientation":"h",
        "yanchor":"bottom",
        "y":-0.3,
        "xanchor":"center",
        "x":0.5
    }
)

fig.show()

/home/erick/projects/study_case_siemens/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [ ]:
business_line_adherence = (
    df_merged.groupby(
        F.col("business_line"),
        F.col("target_srs_ready"),
    ).agg(
        F.count("*").alias("total_count"),
        F.sum(
            F.when(
                F.col("level_of_connection") == "SRS READY",
                F.lit(1)
            ).otherwise(F.lit(0))
        ).alias("srs_ready_count")
    )
    .withColumn("adherence_pct", F.round((F.col("srs_ready_count") / F.col("total_count")*100)))
)
business_line_adherence = (
    business_line_adherence.withColumn(
        "target_srs_ready",
        F.round(F.col("target_srs_ready")*100))
        .orderBy(F.col("adherence_pct").desc())
    )

business_line_adherence.show()

business_line_adherence_pd = business_line_adherence.toPandas()

+-------------+----------------+-----------+---------------+-------------+
|business_line|target_srs_ready|total_count|srs_ready_count|adherence_pct|
+-------------+----------------+-----------+---------------+-------------+
|     MI SPECT|            87.0|        101|             89|         88.0|
|          AXA|            87.0|        183|            161|         88.0|
|           MR|            87.0|        663|            576|         87.0|
|       MI PET|            87.0|         48|             41|         85.0|
|           CT|            87.0|        541|            458|         85.0|
|           SY|            87.0|        150|            115|         77.0|
|        XP WH|            87.0|        114|             85|         75.0|
|        XP RF|            87.0|         67|             50|         75.0|
|           US|            25.0|        320|             58|         18.0|
|       AXA SU|            25.0|         73|             12|         16.0|
|        XP SU|          

/home/erick/projects/study_case_siemens/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [208]:

import plotly.graph_objects as go

bar_colors = [
    "#009999" if row["adherence_pct"] >= row["target_srs_ready"] else "#D05353"
    for _, row in business_line_adherence_pd.iterrows()
]

business_line_adherence_fig = go.Figure()

business_line_adherence_fig.add_trace(
    go.Bar(
        x=business_line_adherence_pd.business_line,
        y=business_line_adherence_pd.adherence_pct,
        name="Aderente (%)",
        marker_color=bar_colors, # Cor das barras
        texttemplate="%{y}",    # Exibe os números nas barras
        textposition="inside",
    )
)

business_line_adherence_fig.add_trace(
    go.Scatter(
        x=business_line_adherence_pd.business_line,
        y=business_line_adherence_pd.target_srs_ready,
        name="Target (%)",
        mode="lines+markers+text", # Linha, pontos e rótulos de texto
        line=dict(color="#ec6602", width=3), # Cor e espessura da linha
        texttemplate="%{y}%",   # Exibe a porcentagem nos pontos
        textposition="top left",
        marker={
            "symbol":"diamond-open",
            "size":10,
            "color":"#ec6602",
            "line":{"width":2}
        }
    )
)

business_line_adherence_fig.update_layout(
    legend={
        "orientation":"h",
        "yanchor":"bottom",
        "y":-0.3,
        "xanchor":"center",
        "x":0.5
    }
)

business_line_adherence_fig.update_yaxes(visible=True)


business_line_adherence_fig.show()

In [ ]:
{
    "service_region": "SÃO PAULO INTERIOR",
    "business_line": "AXA",
    "level_of_connection": "SRS READY"
}

In [215]:
equipment_for_srs_activation = (
    df_merged.withColumn(
    "city/state",
    F.concat(F.col("city"), F.lit("/"), F.col("state"))
    )
    .filter(F.col("level_of_connection") == "CONNECTED BUT NOT SRS READY")
    .select(
        F.col("equipment_serial_number").alias("Serial Number"),
        F.col("t1.system_name").alias("System"),
        F.col("business_line").alias("Business Line"),
        F.col("city/state").alias("City/State"),
        F.col("service_region").alias("Service Region"),
        F.col("current_connectivity_status").alias("Status"),
    )
)
equipment_for_srs_activation.show()

+-------------+-----------+-------------+-----------------+------------------+--------------------+
|Serial Number|     System|Business Line|       City/State|    Service Region|              Status|
+-------------+-----------+-------------+-----------------+------------------+--------------------+
|   3049584789|    ARTIS 1|          AXA|       LIMEIRA/SP|SÃO PAULO INTERIOR|       Connection OK|
|   3049344350|    ARTIS 2|          AXA|         BELEM/PA|            CENTRO|       Connection OK|
|   3004146996| MAGNETOM 9|           MR|     SAO PAULO/SP|         SÃO PAULO|       Connection OK|
|   3026908206|    ARTIS 1|          AXA|RIO DE JANEIRO/RJ|    RIO DE JANEIRO|Connection Expire...|
|   3007640267|  SOMATOM 6|           CT|     GUARULHOS/SP|         SÃO PAULO|Connection Expire...|
|   2100024948| NUCLEAR 14|     MI SPECT|  PORTO ALEGRE/RS|               SUL|       Connection OK|
|   3034590861|MAGNETON 10|           MR|      BRASILIA/DF|            CENTRO|       Connection OK|
